# Поиск 50 релевантных объявлений Avito

Решение состоит из трёх этапов: **BM25 + E5 -> CatBoost -> BGE**
Сначала я совершаю отбор кандидатов по совпадению слов (BM25) и смысловой близости (E5), затем учитываю текстовые и табличные признаки (Catboost), после чего я объединяю его ранжирование с результатом гибридного поиска и выбираю 200 объявлений. В конце я объединяю это ранжирование с оценкой реранкера (BGE) и получаю итоговые 50 объявлений

В этом ноутбуке все эмбеддинги вычисляются заново и CatBoost тоже обучается заново,
E5 и BGE используются с готовыми открытыми весами, без дообучения

## 1. Подготовка

Для запуска нужны Python 3.12, pytorch с CUDA и исходные файлы в папке `dataset`:
`train.parquet`, `benchmark_items.parquet`, `benchmark_queries.parquet`.
Список версий пакетов находится в `requirements_run.txt`

Веса моделей можно либо скачать заново, оставив `DOWNLOAD_MODELS = True` в следующей ячейке. Либо для локального запуска можно поставить `DOWNLOAD_MODELS = False`, если уже есть папка `models` со скачанными весами

После установки пакетов и запуска ноутбука, результаты появятся в папке `avito_train_outputs`

In [2]:
DOWNLOAD_MODELS = True #Если не хотите качать веса моделей заново, то False

if DOWNLOAD_MODELS:
    from huggingface_hub import snapshot_download

    models = [ #тут задаем имена моделей, хеш коммита для воспроизводимости и папку, куда модели будут скачены
        ('intfloat/multilingual-e5-base', 'd128750597153bb5987e10b1c3493a34e5a4502a', 'models/e5'), 
        ('BAAI/bge-reranker-v2-m3', '953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e', 'models/bge'),
    ]
    for name, revision, folder in models:
        snapshot_download(
            name, revision=revision, local_dir=folder,
            allow_patterns=['*.json', '*.safetensors', '*.model', '*.txt'], #лишние файлы не качаем
        )
    print('Веса скачены. Дальше модели работают локально')


/home/vlad/miniconda3/envs/avito_test/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 6 files: 100%|████████████████████████| 6/6 [00:00<00:00, 1750.42it/s]

Веса скачены. Дальше модели работают локально


In [3]:
import os
#Запрещаем сетевые обращения для моделей
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
#Импортим библиотеки
from pathlib import Path
from functools import lru_cache
import gc
import hashlib
import html
import json
import re
import unicodedata

import numpy as np
import pandas as pd
import bm25s
import Stemmer
import faiss
from catboost import CatBoostRanker, Pool
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm
#Задаем пути
data_dir = Path("dataset")
output_dir = Path("avito_train_outputs")
output_dir.mkdir(exist_ok=True)
seed = 46 #для воспроизводимости
pd.set_option("display.max_colwidth", 90)

RUN_VALIDATION = True #Это тумблер для локальной проверки качества. Можно поставить False, если нужен только answer.csv (на ответ не влияет)
E5_PATH = Path('models/e5')
BGE_PATH = Path('models/bge')
assert all((data_dir / name).is_file() for name in
           ['train.parquet', 'benchmark_items.parquet', 'benchmark_queries.parquet'])
assert (E5_PATH / 'config.json').is_file() and (BGE_PATH / 'config.json').is_file(), 'Подготовьте веса по инструкции выше'

## 2. Данные и разметка

В `train` задаётся положительную пара запрос - объявление\
В `benchmark_queries`задаются запросы без разметки\
В `benchmark_items` объявления

In [4]:
query_columns = [
    "search_query", "search_location_id", "search_is_delivery_search",
    "search_infm_params_text", "search_category",
]

#Здесь неполный набор фичей остался от экспериментов
item_columns = [
    "item_id", "item_title_raw", "item_description_raw", "item_infm_params_text",
    "item_location_id", "item_category_id", "item_microcat_id",
    "item_rating", "item_rating_reviews_count",
]
#Считываем данные
train = pd.read_parquet(data_dir / "train.parquet", columns=query_columns + ["item_id"])
benchmark_queries = pd.read_parquet(data_dir / "benchmark_queries.parquet")
benchmark_items = pd.read_parquet(data_dir / "benchmark_items.parquet", dtype_backend="pyarrow") #pyarrow для скорости, чтобы не создавать лишних копий

assert benchmark_items.item_id.is_unique #Проверка на дубли
assert benchmark_queries.query_id.is_unique #Проверка на дубли

display(train.head(3))
print("Train:", len(train), "Запросов для ответа:", len(benchmark_queries))

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_id
0,скупка телевизоров,652430,0,,114,e8b685dffe1a408e
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,92e1b0446f827b59
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье",114,624846856ce81d69


Train: 497673 Запросов для ответа: 2452


Задаем новые признаки:\
`query_group` - нормализованный текст, нужен для разделения выборок без одинаковых текстов\
`query_key` - текст вместе с локацией, категорией и фильтрами. Один текст в разных городах это разные контексты

In [5]:
def normalize_query(text):
    text = unicodedata.normalize("NFKC", str(text)).lower().replace("ё", "е") # нормализую текст, привожу к нижнему регистру и меняю везде "ё" на "е"
    return re.sub(r"\s+", " ", text).strip() #Схлопываю  пробелы


def add_query_keys(frame):
    frame = frame.copy()
    frame["query_group"] = frame.search_query.map(normalize_query) #нормализация к каждой строке запроса
    context = frame[query_columns].copy() # Задаю контекст
    context["search_query"] = frame.query_group
    context["search_infm_params_text"] = context.search_infm_params_text.fillna("").str.strip()
    frame["query_key"] = pd.util.hash_pandas_object(context, index=False).astype("uint64") # Считаю хеш по каждой строке контекста
    assert frame.query_key.nunique() == len(context.drop_duplicates()) #Проверка на коллизию хешей
    return frame


train = add_query_keys(train)
benchmark_queries = add_query_keys(benchmark_queries)

#удаляем дубликаты, чтобы все запросы и запросы-ответы были уникальны
queries = train[query_columns + ["query_key", "query_group"]].drop_duplicates("query_key")
queries = queries.reset_index(drop=True) 
positives = train[["query_key", "item_id"]].drop_duplicates().reset_index(drop=True)

print("Удалено повторных связей:", len(train) - len(positives))
display(queries[["search_query", "search_location_id", "query_group", "query_key"]].head(7))

Удалено повторных связей: 30644


,search_query,search_location_id,query_group,query_key
0,скупка телевизоров,652430,скупка телевизоров,14335288334562339031
1,автоподбор,640860,автоподбор,733677169235430385
2,баня на дровах,653240,баня на дровах,13448624250200791775
3,изготовление госномера на авто,634670,изготовление госномера на авто,16875324630120112311
4,укладка плитки,658430,укладка плитки,4625857356279998968
5,сборка мебели,639550,сборка мебели,15768112961906960894
6,наращивание ногтей,107620,наращивание ногтей,42988392970325329


Объединяю объявления `benchmark` и `train`. Сначала идут `benchmark`, при совпадении `ID` используются
их поля. 
Дополнительные поля, которые тут не читаются, будут прочитаны позже перед построением признаков

На основной проверке ищем во всём объединённом корпусе. На дополнительной и при формировании
ответа поиск только среди `benchmark` объявлений

In [6]:
train_items = pd.read_parquet(
    data_dir / "train.parquet", columns=item_columns, dtype_backend="pyarrow"
)

train_items = train_items.drop_duplicates("item_id")
train_items = train_items.loc[~train_items.item_id.isin(benchmark_items.item_id)] #защита от утечки данных
corpus = pd.concat([benchmark_items[item_columns], train_items], ignore_index=True)

del train_items #удаляю промежуточный дф
gc.collect()

assert corpus.item_id.is_unique #проверка корпуса на уникальность
print("Всего объявлений:", len(corpus))
display(corpus[["item_id", "item_title_raw", "item_location_id"]].head(7))

Всего объявлений: 515895


,item_id,item_title_raw,item_location_id
0,111eb8b979577d79,Ремонт/выкуп компьют. и ноутбуков с выездом на дом,631060
1,75fc8e10f5a66fc4,Обучение ребенка чтению,631870
2,3f6ae81704565b9c,Афрокудри 5+,628500
3,0618dec37a59a21a,Монтаж малых архитектурных форм,637640
4,13da2c81574677ed,Репетитор по истории 10 класс ЕГЭ,624850
5,285fdc0b02a11b17,Ремонт посудомоечных машин с выездом на дом,637640
6,ac2beb33ad106198,Подарок на праздник - Песня на заказ,662150


In [7]:
# Выгружаю колонки корпуса в numpy массивы и строю вспомогательные структуры для поиска: маски допустимых товаров (весь корпус / только бенчмарк),
# группировку по локации и сортировку по популярности. Затем собираю словари правильных
# ответов (answers_all и answers_benchmark) для подсчёта метрик
item_ids = corpus.item_id.astype(str).to_numpy()
item_locations = corpus.item_location_id.to_numpy(dtype=np.int64)
reviews = corpus.item_rating_reviews_count.fillna(0).to_numpy(dtype=np.float64)
titles = corpus.item_title_raw.fillna("").to_numpy(dtype=object)
params = corpus.item_infm_params_text.fillna("").to_numpy(dtype=object)
categories = corpus.item_category_id.fillna(-1).to_numpy(dtype=np.int64)
ratings = pd.to_numeric(corpus.item_rating, errors="coerce").to_numpy(dtype=np.float32, na_value=np.nan)

doc_rows = np.arange(len(corpus), dtype=np.int32)
allowed_all = np.ones(len(corpus), dtype=bool)
allowed_benchmark = doc_rows < len(benchmark_items)
location_rows = pd.Series(doc_rows).groupby(item_locations, sort=False).indices
popular_rows = np.lexsort((item_ids, -reviews))

positives["doc_idx"] = pd.Index(item_ids).get_indexer(positives.item_id)
assert positives.doc_idx.ge(0).all()
answers_all = positives.groupby("query_key", sort=False).doc_idx.apply(np.array).to_dict()
answers_benchmark = {
    key: rows[allowed_benchmark[rows]]
    for key, rows in answers_all.items() if allowed_benchmark[rows].any()
}

## 3. Обучение и валидация

Валидация содержит 3000 запросов для сравнения новой модели с предыдущей на тех же запросах \
В обучение идут все остальные тексты. Если текст попал в валидацию, исключаю из обучения все его города, категории и фильтры, чтобы посмотреть, как модель ведет себя с текстами запросов, которые она не видела при обучении. Точные повторные пары уже удалены выше

In [8]:
def stable_hash(value, salt):
    text = f"{seed}|{salt}|{value}"
    return int.from_bytes(hashlib.blake2b(text.encode(), digest_size=8).digest(), "little")


# Эти метки нужны только для воспроизведения прежней валидациолнной выборки
queries["context_order"] = queries.query_key.map(lambda key: stable_hash(int(key), "context"))

In [9]:
def fixed_validation(queries):
    # Детерминированно восстанавливаем прежние 3000 валидационных запроса без изменения состава
    candidates = queries.loc[
        queries.query_group.map(lambda text: stable_hash(text, "holdout") % 5 != 0)
    ].copy()
    candidates["fold"] = candidates.query_group.map(lambda text: stable_hash(text, "fold") % 3)
    candidates = candidates.sort_values("context_order").drop_duplicates("query_group")
    parts = [part.sample(n=min(1000, len(part)), random_state=seed + int(fold))
             for fold, part in candidates.groupby("fold", sort=True)]
    return pd.concat(parts, ignore_index=True)


validation_queries = fixed_validation(queries)
validation_texts = set(validation_queries.query_group)
# Все оставшиеся контексты, включая разные города и фильтры одного текста
train_queries = queries.loc[~queries.query_group.isin(validation_texts)].copy()
train_queries = train_queries.sort_values(["context_order", "query_key"]).reset_index(drop=True)
#Проверяем размер валидации, уникальность внутри выборок, отсутствие утечек и полноту представления текстов
assert len(validation_queries) == 3000
assert validation_queries.query_group.is_unique
assert train_queries.query_key.is_unique
assert not set(train_queries.query_group) & validation_texts
assert set(train_queries.query_group) | validation_texts == set(queries.query_group)
assert len(train_queries) == (~queries.query_group.isin(validation_texts)).sum()

In [10]:
split_summary = pd.DataFrame([
    {"set": "train", "texts": train_queries.query_group.nunique(), "contexts": len(train_queries)},
    {"set": "validation", "texts": len(validation_texts), "contexts": len(validation_queries)},
])
display(split_summary)

# Сохраняем точный состав, чтобы последующие сравнения использовали ту же проверку
split_summary.to_csv(output_dir / 'split_summary.csv', index=False)
validation_queries[["query_key", "query_group"]].to_csv(output_dir / 'validation_queries.csv', index=False)

,set,texts,contexts
0,train,70873,339616
1,validation,3000,3000


## 4. Поиск по словам: BM25

BM25 оценивает совпадения слов, учитывая их редкость и длину документа. Удаляю HTML, привожу текст к нижнему регистру и выделяю основы слов через pystemmer. Создаю два индекса: заголовок с параметрами (до 160 слов) и описание (до 200 слов). Для заголовка ограничение скорее формальное, для описания ограничение связано с тем, чтобы уменьшить влияние длинных рекламных объявлений. При большем количестве времени, стоило бы исследовать этот выбор на оптимальность. Их баллы складываются

In [11]:
stemmer = Stemmer.Stemmer("russian")
word_pattern = r"(?u)\b\w+\b"


def prepare_text(text, limit=None):
    if text is None or pd.isna(text):
        return ""
    text = re.sub(r"<[^>]+>", " ", html.unescape(str(text)))
    words = re.findall(word_pattern, normalize_query(text))
    return " ".join(words[:limit])


@lru_cache(maxsize=20000) # Чтобы не пересчитывалось для частых поисковых запросов
def query_tokens(text):
    return tuple(stemmer.stemWords(prepare_text(text).split()))


print(prepare_text("<b>Ремонт телефонов!</b> Быстро"))
print(query_tokens("ремонт телефонов"))

ремонт телефонов быстро
('ремонт', 'телефон')


In [12]:
# Строим индексы BM25
def build_bm25(texts, word_limit):
    cleaned = [prepare_text(text, word_limit) for text in tqdm(texts, desc="Тексты BM25")]
    tokens = bm25s.tokenize(
        cleaned, lower=False, stopwords=[], stemmer=stemmer.stemWords,
        token_pattern=word_pattern, show_progress=False,
    )
    
    index = bm25s.BM25(k1=1.2, b=0.75, method="lucene", backend="numpy")
    index.index(tokens, show_progress=True)
    
    return index


head_index = build_bm25((title + " " + par for title, par in zip(titles, params)), 160)
description_index = build_bm25(corpus.item_description_raw.fillna(""), 200)

Тексты BM25: 515895it [00:28, 17881.25it/s]
Тексты BM25: 100%|██████████████████| 515895/515895 [00:33<00:00, 15432.91it/s]


In [13]:
def bm25_scores(index, tokens):
    if not tokens:
        return np.zeros(len(corpus), dtype=np.float32)
    return index.get_scores(list(tokens))

#Проверка работы скоров
example = validation_queries.iloc[0] 
tokens = query_tokens(example.search_query)
example_bm25 = bm25_scores(head_index, tokens) + bm25_scores(description_index, tokens) #Складываем оценки
print("Запрос:", example.search_query)
print("Локация:", example.search_location_id)

Запрос: фото на документы в форме
Локация: 621540


Тут беру до 500 объявлений по всему допустимому корпусу и до 500 из локации запроса. Локация не является жёстким фильтром, глобальный поиск остаётся. При нехватке кандидатов добавляю объявления по числу отзывов. При одинаковых баллах использую число отзывов, затем ID это делает порядок устойчивым

In [14]:
def order_by_score(rows, scores): #Сортировка по описанию выше
    return np.lexsort((item_ids[rows], -reviews[rows], -scores))


def top_bm25(scores, rows, k=500): #выбор top k
    rows = np.asarray(rows, dtype=np.int32)
    rows = rows[scores[rows] > 0]
    
    if len(rows) > k:
        cutoff = np.partition(scores[rows], len(rows) - k)[len(rows) - k]
        better = rows[scores[rows] > cutoff]
        tied = rows[scores[rows] == cutoff]
        tied = tied[order_by_score(tied, scores[tied])[:k - len(better)]]
        rows = np.concatenate([better, tied])
        
    return rows[order_by_score(rows, scores[rows])]


example_top = top_bm25(example_bm25, doc_rows)[:5]
display(corpus.iloc[example_top][["item_title_raw", "item_location_id"]].assign(bm25=example_bm25[example_top]))

,item_title_raw,item_location_id,bm25
385706,Фото на документы в форме онлайн,629400,17.623207
97909,Фото на документы онлайн по селфи,635860,15.883251
139763,Фото на документы онлайн по селфи,656830,15.444204
2023,Фото на документы онлайн по селфи,662210,15.313290
53447,Подстановка различной формы для фото на документов,635320,15.156294


In [15]:
def lexical_candidates(scores, location, allowed): # Строю пул кандидатов для запроса
    local_rows = np.asarray(location_rows.get(int(location), []), dtype=np.int32)
    local_rows = local_rows[allowed[local_rows]]
    global_top = top_bm25(scores, doc_rows[allowed]) # Топ глобальный, необязательно в нужной локации
    local_top = top_bm25(scores, local_rows) # Топ в нужной локации
    
    pool = np.union1d(global_top, local_top)
    if len(pool) < 50: # Если кандидатов мало, добиваю популярными
        local_popular = local_rows[np.lexsort((item_ids[local_rows], -reviews[local_rows]))[:25]]
        global_popular = popular_rows[allowed[popular_rows]][:100]
        fallback = list(dict.fromkeys(np.concatenate([local_popular, global_popular])))[:50]
        pool = np.union1d(pool, fallback)
        
    return global_top, local_top, pool.astype(np.int32)


_, _, example_pool = lexical_candidates(example_bm25, example.search_location_id, allowed_all)
print("Кандидатов BM25:", len(example_pool))

Кандидатов BM25: 500


## 5. Поиск по смыслу: E5

Модель [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base) кодирует запросы и объявления в векторы из 768 чисел. Используются предобученные веса, в коде она не обучается. Используем префиксы `query:` и `passage:`, усредняем векторы непустых токенов и нормируем результат. Сохраняем настройки эксперимента: E5 в `float16`, batch size 32, до 256 токенов на объявление и 64 на запрос. Блоки по 4096 строк и порядок батчей также сохранены. Изменение этих параметров может немного менять результат вычислений на GPU

In [16]:
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

assert torch.cuda.is_available(), 'Без cudы не получится :)'

device = 'cuda'
dtype = torch.float16
batch_size = 32
model_revision = 'd128750597153bb5987e10b1c3493a34e5a4502a'
bge_revision = '953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e'
print('GPU:', torch.cuda.get_device_name(0), 'E5:', dtype)

tokenizer = AutoTokenizer.from_pretrained(E5_PATH, local_files_only=True, use_fast=True)
encoder = AutoModel.from_pretrained(E5_PATH, local_files_only=True, torch_dtype=dtype, use_safetensors=True).to(device).eval()

GPU: NVIDIA GeForce RTX 5060 Ti E5: torch.float16


`torch_dtype` is deprecated! Use `dtype` instead!


In [17]:
def clean_dense_text(text): #очистка текста, почти то же самое, что из лексической части, но без нормализации и стемминга
    if text is None or pd.isna(text):
        return ""
    text = re.sub(r"<[^>]+>", " ", html.unescape(str(text)))
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", text)).strip()


def texts_for_e5(frame, kind): # форматирование под e5
    if kind == "queries":
        result = []
        for query in frame.itertuples(index=False):
            text = normalize_query(clean_dense_text(query.search_query))
            filters = clean_dense_text(query.search_infm_params_text)
            result.append("query: " + text + ("\n" + filters if filters else ""))
        return result
        
    fields = []
    for column, limit in [("item_title_raw", 64), ("item_infm_params_text", 48)]:
        texts = [clean_dense_text(value) for value in frame[column]]
        ids = tokenizer(texts, add_special_tokens=False, truncation=True, max_length=limit)["input_ids"]
        fields.append(tokenizer.batch_decode(ids, skip_special_tokens=True))
    descriptions = frame.item_description_raw.map(clean_dense_text)
    
    return ["passage: " + "\n".join(part for part in parts if part)
            for parts in zip(fields[0], fields[1], descriptions)]

In [18]:
def encode_texts(texts, max_length): # энкодим тексты
    order = np.argsort([-len(text) for text in texts], kind="stable")
    vectors = np.empty((len(texts), 768), dtype=np.float32)
    
    for start in range(0, len(texts), batch_size):
        rows = order[start:start + batch_size]
        tokens = tokenizer([texts[i] for i in rows], padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
        
        with torch.inference_mode():
            hidden = encoder(**tokens).last_hidden_state.float()
            hidden = hidden.masked_fill(~tokens.attention_mask[..., None].bool(), 0.0)
            pooled = hidden.sum(dim=1) / tokens.attention_mask.sum(dim=1).clamp_min(1)[:, None]
            normalized = torch.nn.functional.normalize(pooled, p=2, dim=1)
        vectors[rows] = normalized.cpu().numpy()

    return vectors

In [19]:
def encode_frame(frame, kind, name): # Пересчитываем все блоки, файлы прежних запусков не читаем
    result = np.empty((len(frame), 768), dtype=np.float32)
    
    for start in tqdm(range(0, len(frame), 4096), desc = name):
        end = min(start + 4096, len(frame))
        texts = texts_for_e5(frame.iloc[start:end], kind)
        result[start:end] = encode_texts(texts, 256 if kind == 'documents' else 64)
        
    return result


doc_vectors = encode_frame(corpus, 'documents', 'Объявления E5')

assert doc_vectors.shape == (len(corpus), 768)

Объявления E5: 100%|█████████████████████████| 126/126 [11:43<00:00,  5.58s/it]


In [20]:
#Тут кодирую все запросы в эмбеддинги через E5 и собираю их в словарь vector_by_query, заодно освобождаю память
base_queries = pd.concat([validation_queries, benchmark_queries], ignore_index=True)
base_queries = base_queries.drop_duplicates("query_key").sort_values("query_key").reset_index(drop=True)

training_queries = train_queries.loc[~train_queries.query_key.isin(base_queries.query_key)]
training_queries = training_queries.sort_values("query_key").reset_index(drop=True)

base_vectors = encode_frame(base_queries, "queries", "validation_and_benchmark")
training_vectors = encode_frame(training_queries, "queries", "training_queries")

vector_by_query = dict(zip(base_queries.query_key, base_vectors))
vector_by_query.update(zip(training_queries.query_key, training_vectors))

del encoder
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

training_queries: 100%|████████████████████████| 83/83 [00:50<00:00,  1.64it/s]


Тут строю FAISS индекс для поиска ближайших товаров по эмбеддингам и для каждого запроса нахожу top 500

In [21]:
faiss.omp_set_num_threads(4)
blas_limit = threadpool_limits(limits=4, user_api="blas")


def nearest_documents(query_vectors, rows):
    result = np.full((len(query_vectors), 500), -1, dtype=np.int32)
    if not len(rows):
        return result
        
    index = faiss.IndexFlatIP(doc_vectors.shape[1])
    for start in range(0, len(rows), 8192):
        index.add(np.ascontiguousarray(doc_vectors[rows[start:start + 8192]], dtype=np.float32))
    k = min(500, len(rows))
    
    for start in range(0, len(query_vectors), 128):
        scores, positions = index.search(np.ascontiguousarray(query_vectors[start:start + 128]), k)
        ids = rows[positions]
        for offset in range(len(ids)):
            order = np.lexsort((ids[offset], -scores[offset]))
            result[start + offset, :k] = ids[offset, order]
    return result

In [22]:
# Для каждого запроса нахожу top 500 ближайших товаров по эмбеддингам, отдельно глобально и отдельно по локации
def semantic_candidates(frame, allowed):
    vectors = np.asarray([vector_by_query[key] for key in frame.query_key], dtype=np.float32)
    global_ids = nearest_documents(vectors, doc_rows[allowed])
    local_ids = np.full_like(global_ids, -1)
    groups = frame.groupby("search_location_id", sort=False).indices
    
    for location, query_rows in tqdm(groups.items(), desc="Локации E5"):
        rows = np.asarray(location_rows.get(int(location), []), dtype=np.int32)
        local_ids[query_rows] = nearest_documents(vectors[query_rows], rows[allowed[rows]])
        
    return global_ids, local_ids


example_frame = validation_queries.iloc[[0]].reset_index(drop=True)
example_global, example_local = semantic_candidates(example_frame, allowed_all)
display(corpus.iloc[example_global[0, :5]][["item_title_raw", "item_location_id"]])

Локации E5: 100%|█████████████████████████████| 1/1 [00:00<00:00, 21183.35it/s]


,item_title_raw,item_location_id
385706,Фото на документы в форме онлайн,629400
11165,Фото на документы /ретушь/ подставление формы,641780
58663,Фотограф,637640
111166,"Фото на документы онлайн, идеальная ретушь фото",637640
11617,"Фотография на документы, фотоуслуги, печать",642320


Здесь происходит финальная сборка пула кандидатов для одного запроса: объединяю результаты лексической чавсти BM25 и семантической E5 в один массив и возвращаю его вместе со скорами и разбивкой по каналам (какие кандидаты пришли откуда)

In [23]:
def find_candidates(query, dense_global, dense_local, allowed):
    tokens = query_tokens(query.search_query)
    head = bm25_scores(head_index, tokens)
    body = bm25_scores(description_index, tokens)
    lex_global, lex_local, lex_pool = lexical_candidates(head + body, query.search_location_id, allowed)
    
    dense_global = dense_global[dense_global >= 0]
    dense_local = dense_local[dense_local >= 0]
    rows = np.unique(np.concatenate([lex_pool, dense_global, dense_local])).astype(np.int32)

    channels = [lex_global, lex_local, dense_global, dense_local]
    
    return rows, head[rows], body[rows], channels

## 6. Признаки для CatBoost

Сначала получаю исходное гибридное ранжирование. У BM25 и E5 разные шкалы, поэтому объединяю их, используя RRF: вклад позиции `r` равен `1 / (60 + r)` \
Совпадение локации увеличивает BM25 в 4 раза и добавляет 0.03 к косинусному сходству перед ранжированием(небольшой бонус за совпадение локации, оптимальность конкретно такой добавки не проверялась)\
Признаки включают баллы и места поисковых методов, совпадения слов, чисел и фильтров, категорию, локацию, рейтинг и число отзывов. `reference_rank` это место в гибриде

In [24]:
def ranks_by_score(rows, scores):
    order = order_by_score(rows, scores)
    ranks = np.empty(len(rows), dtype=np.int32)
    ranks[order] = np.arange(1, len(rows) + 1)
    return ranks


def channel_rrf(rows, channel):
    values = np.zeros(len(rows), dtype=np.float32)
    values[np.searchsorted(rows, channel)] = 1.0 / (60 + np.arange(1, len(channel) + 1))
    return values

In [25]:
def score_features(query, rows, head, body, channels):
    cosine = np.asarray(doc_vectors[rows], dtype=np.float32) @ vector_by_query[query.query_key]
    same_location = item_locations[rows] == int(query.search_location_id)
    
    lexical = head + body
    lexical_geo = lexical * (1.0 + 3.0 * same_location)
    cosine_geo = cosine + 0.03 * same_location
    lex_rrf = np.where(lexical_geo > 0, 1.0 / (60 + ranks_by_score(rows, lexical_geo)), 0.0)
    dense_rrf = 1.0 / (60 + ranks_by_score(rows, cosine_geo))
    fusion = lex_rrf + dense_rrf
    
    data = pd.DataFrame({
        "bm25_head": head, "bm25_description": body, "bm25_total": lexical,
        "bm25_geo": lexical_geo, "bm25_relative": lexical / max(float(lexical.max()), 1e-8),
        "cosine": cosine, "cosine_geo": cosine_geo, "cosine_gap_to_best": cosine - cosine.max(),
        "lexical_rrf": lex_rrf, "dense_rrf": dense_rrf, "fusion_rrf": fusion,
        "same_location": same_location,
    })
    names = ["lex_global_rrf", "lex_local_rrf", "dense_global_rrf", "dense_local_rrf"]
    
    for name, channel in zip(names, channels):
        data[name] = channel_rrf(rows, channel)
    data["doc_idx"] = rows
    data["reference_rank"] = ranks_by_score(rows, fusion)
    
    return data

In [26]:
# Демонстрация полного пайплайна на одном запросе
example_rows, head, body, channels = find_candidates(example, example_global[0], example_local[0], allowed_all)
example_features = score_features(example, example_rows, head, body, channels)
example_view = example_features.assign(title=titles[example_rows])
display(example_view.sort_values("reference_rank").head(10)[
    ["title", "bm25_total", "cosine", "same_location", "reference_rank"]
])
print("Кандидатов после объединения:", len(example_rows))

,title,bm25_total,cosine,same_location,reference_rank
690,Фото на документы в форме онлайн,17.623207,0.896049,False,1
816,"Красивые фото на документы, на паспорт онлайн",14.914305,0.875972,False,2
25,Фото на документы /ретушь/ подставление формы,13.437117,0.890596,False,3
143,Подстановка различной формы для фото на документов,15.156294,0.873230,False,4
60,Фото на документы от профессионала,13.455994,0.874819,False,5
271,"Фото на документы онлайн, идеальная ретушь фото",11.365733,0.881317,False,6
855,Фото на паспорт и документы,14.274690,0.871032,False,7
292,Фото на документы. Ретушь фотографий,10.469315,0.879894,False,8
341,Фото на документы онлайн по селфи,15.444204,0.867181,False,9
179,"Фото на документы,реставрация,фотошоп,ксерокс и др",10.446288,0.878002,False,10


Кандидатов после объединения: 864


In [27]:
@lru_cache(maxsize=50000)
def text_parts(text): #Подготовка текста для признаков
    cleaned = prepare_text(text)
    words = cleaned.split()
    stems = frozenset(stemmer.stemWords(words))
    numbers = frozenset(re.findall(r"\d+", cleaned))
    
    return cleaned, stems, numbers, len(words)

In [28]:
def add_text_features(data, query): #текстовые фичи на основе пересечения множеств между запросом и каждым кандидатом
    query_text, query_words, query_numbers, query_length = text_parts(str(query.search_query))
    _, filter_words, _, _ = text_parts(clean_dense_text(query.search_infm_params_text))
    values = []
    
    for row in data.doc_idx:
        title, title_words, title_numbers, title_length = text_parts(titles[row])
        _, param_words, param_numbers, _ = text_parts(params[row])
        head_words = title_words | param_words
        
        values.append([
            len(query_words & title_words) / max(1, len(query_words)),
            len(query_words & head_words) / max(1, len(query_words)),
            len(filter_words & head_words) / max(1, len(filter_words)),
            bool(query_text) and " " + query_text + " " in " " + title + " ",
            len(query_numbers & (title_numbers | param_numbers)) / max(1, len(query_numbers)),
            title_length,
        ])
        
    columns = ["title_coverage", "head_coverage", "filter_coverage",
               "query_phrase_in_title", "number_coverage", "title_words"]
    data[columns] = np.asarray(values, dtype=np.float32)
    data["query_words"] = query_length
    data["query_characters"] = len(query_text)
    data["has_filters"] = bool(filter_words)
    data["has_numbers"] = bool(query_numbers)

In [29]:
def add_metadata_features(data, query): #добавляем метаданные фичи
    rows = data.doc_idx.to_numpy()
    rating = ratings[rows]
    data["category_match"] = (int(query.search_category) != 0) & (categories[rows] == int(query.search_category))
    data["category_unspecified"] = int(query.search_category) == 0
    data["delivery_search"] = bool(query.search_is_delivery_search)
    data["rating"] = np.where(np.isfinite(rating), rating, 0.0)
    data["rating_missing"] = ~np.isfinite(rating)
    data["log_reviews"] = np.log1p(np.maximum(reviews[rows], 0)).astype(np.float32)
    data["log_pool_size"] = np.log1p(len(rows))

In [30]:
feature_names = [
    "bm25_head", "bm25_description", "bm25_total", "bm25_geo", "bm25_relative",
    "cosine", "cosine_geo", "cosine_gap_to_best", "lexical_rrf", "dense_rrf", "fusion_rrf",
    "lex_global_rrf", "lex_local_rrf", "dense_global_rrf", "dense_local_rrf",
    "same_location", "category_match", "category_unspecified", "delivery_search",
    "query_words", "query_characters", "has_filters", "title_coverage", "head_coverage",
    "filter_coverage", "query_phrase_in_title", "number_coverage", "has_numbers",
    "title_words", "rating", "rating_missing", "log_reviews", "log_pool_size",
]

#финальная сборка всех фичей в одну таблицу, объединяет базовые фичи, текстовые и метаданные
def make_features(query, dense_global, dense_local, allowed):
    rows, head, body, channels = find_candidates(query, dense_global, dense_local, allowed)
    data = score_features(query, rows, head, body, channels)
    add_text_features(data, query)
    add_metadata_features(data, query)
    data[feature_names] = data[feature_names].astype(np.float32)
    assert np.isfinite(data[feature_names].to_numpy()).all()
    return data


example_features = make_features(example, example_global[0], example_local[0], allowed_all)
display(example_features.sort_values("reference_rank").head(5)[feature_names])

,bm25_head,bm25_description,bm25_total,bm25_geo,bm25_relative,cosine,cosine_geo,cosine_gap_to_best,lexical_rrf,dense_rrf,...,head_coverage,filter_coverage,query_phrase_in_title,number_coverage,has_numbers,title_words,rating,rating_missing,log_reviews,log_pool_size
690,12.206858,5.416350,17.623207,17.623207,1.000000,0.896049,0.896049,0.000000,0.016393,0.016393,...,1.0,1.0,1.0,0.0,0.0,6.0,5.000000,0.0,2.302585,6.76273
816,8.957803,5.956502,14.914305,14.914305,0.846288,0.875972,0.875972,-0.020077,0.015152,0.013514,...,0.8,1.0,0.0,0.0,0.0,7.0,4.964912,0.0,4.744932,6.76273
25,9.640085,3.797031,13.437117,13.437117,0.762467,0.890596,0.890596,-0.005453,0.011494,0.016129,...,0.8,1.0,0.0,0.0,0.0,6.0,5.000000,0.0,5.950643,6.76273
143,10.446128,4.710166,15.156294,15.156294,0.860019,0.873230,0.873230,-0.022819,0.015385,0.010417,...,0.8,1.0,0.0,0.0,0.0,7.0,0.000000,1.0,0.693147,6.76273
60,8.272678,5.183316,13.455994,13.455994,0.763538,0.874819,0.874819,-0.021229,0.012048,0.012048,...,0.8,1.0,0.0,0.0,0.0,5.0,5.000000,0.0,5.849325,6.76273


Здесь загружаю цену, координаты, флаги связи и подкатегорию, выравниваю данные по порядку объявлений в корпусе и обрабатываю пропуски. По корректным координатам вычисляю приблизительный центр каждой локации. Затем задаю список дополнительных признаков катбуст и отмечаю категориальные признаки

In [31]:
columns=['item_id','item_price','item_latitude','item_longitude',
         'item_is_phone_hidden','item_is_message_forbidden','item_microcat_id']

tables=[pd.read_parquet(data_dir/name,columns=columns) for name in ['benchmark_items.parquet','train.parquet']]
extra=pd.concat(tables,ignore_index=True).drop_duplicates('item_id',keep='first')
extra['item_id']=extra.item_id.astype(str)
extra=extra.set_index('item_id')
assert pd.Index(item_ids).isin(extra.index).all()
extra=extra.loc[item_ids].reset_index()
del tables

def numbers(name):
    return pd.to_numeric(extra[name],errors='coerce').to_numpy(dtype=np.float64,na_value=np.nan)
    
prices=numbers('item_price')
latitudes,longitudes=numbers('item_latitude'),numbers('item_longitude')
geo_valid=np.isfinite(latitudes)&np.isfinite(longitudes)&(np.abs(latitudes)<=90)&(np.abs(longitudes)<=180)&~((latitudes==0)&(longitudes==0))
phone_hidden=np.nan_to_num(numbers('item_is_phone_hidden'),nan=-1)
message_forbidden=np.nan_to_num(numbers('item_is_message_forbidden'),nan=-1)
microcategories=extra.item_microcat_id.astype('string').fillna('missing').astype(str).to_numpy()
centers=pd.DataFrame({'location':item_locations[geo_valid], 'lat':latitudes[geo_valid], 'lon':longitudes[geo_valid]}).groupby('location')[['lat','lon']].median()
location_centers={int(k):tuple(v) for k,v in zip(centers.index,centers.to_numpy())}

categorical_features=['microcategory','query_category_microcategory']
added=['price_log','price_missing','price_zero','price_log_vs_pool','price_percentile_pool',
       'phone_hidden','message_forbidden','latitude','longitude','coordinates_missing',
       'distance_to_location_center_log','distance_missing'] + categorical_features
all_features=list(feature_names) + added

In [32]:
def add_extra_features(data, query):
    #Все относительные признаки считаются на полном пуле до выборки негативных пар, чтобы избежать утечки
    result = data.copy()
    rows = data.doc_idx.to_numpy()
    # Признаки цены
    p = prices[rows]
    good = np.isfinite(p) & (p >= 0)
    logs = np.log1p(np.where(good, p, 0))
    result['price_log'] = logs
    result['price_missing'] = ~good
    result['price_zero'] = good & (p == 0)
    med = np.median(logs[good]) if good.any() else 0.0
    result['price_log_vs_pool'] = logs - med
    ranks = pd.Series(np.where(good, p, np.nan)).rank(pct=True).fillna(0).to_numpy()
    result['price_percentile_pool'] = ranks
    #Флаги объявления
    result['phone_hidden'] = phone_hidden[rows]
    result['message_forbidden'] = message_forbidden[rows]
    # Координаты
    lat, lon = latitudes[rows], longitudes[rows]
    valid = geo_valid[rows]
    result['latitude'] = np.where(valid, lat, 0)
    result['longitude'] = np.where(valid, lon, 0)
    result['coordinates_missing'] = ~valid
    #Расстояние до центра локации
    center = location_centers.get(int(query.search_location_id))
    distance = np.zeros(len(rows), dtype=np.float64)
    
    if center is not None:
        a, b = np.radians(lat), np.radians(lon)
        ca, cb = np.radians(center)
        h = np.sin((a-ca)/2)**2 + np.cos(a)*np.cos(ca)*np.sin((b-cb)/2)**2
        distance = 6371 * 2 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))
        
    distance = np.where(valid & (center is not None), distance, 0)
    result['distance_to_location_center_log'] = np.log1p(distance)
    result['distance_missing'] = ~valid | (center is None)
    # Категориальные признаки
    result['microcategory'] = microcategories[rows]
    result['query_category_microcategory'] = [str(query.search_category) + ':' + x for x in microcategories[rows]]
    numeric = [c for c in result.columns if c not in categorical_features and c != 'doc_idx']

    return result

## 7. Обучение CatBoost

Здесь обучаю катбуст на всех контекстах обучающих текстов, для каждого запроса беру все размеченные положительные объявления из пула, 64 неразмеченных объявления с лучшими позициями в гибридном ранжировании и до 64 случайных из оставшихся. Неразмеченные пары считаю отрицательными, хотя из-за неполной разметки среди в них и могут быть положительные пары, если разметка неполная. Запросы без положительных или отрицательных кандидатов пропускаю. Использую катбустранкер с YetiRank на CPU: 100 деревьев глубины 6, learning rate 0.05 и L2 регуляризация 10. Параметры берутся из предыдущих экспериментов. Для экономии памяти записываю обучающие пары порциями, затем загружаю файл в пул. Файл создаётся заново при запуске

In [33]:
def sample_training_pairs(data, query_key): #Формирую обучающую выборку для ранжировщика, негативное семплирование
    labels = np.isin(data.doc_idx, answers_all[query_key])
    positive = np.flatnonzero(labels)
    negative = np.flatnonzero(~labels)
    if not len(positive) or not len(negative):
        return data.iloc[:0].assign(target=np.zeros(0, dtype=np.uint8))
        
    negative = negative[np.argsort(data.reference_rank.to_numpy()[negative], kind="stable")]
    hard = negative[:64]
    remaining = negative[64:]
    rng = np.random.default_rng(stable_hash(int(query_key), "ranker-negatives-v3"))
    random = rng.choice(remaining, size=min(64, len(remaining)), replace=False)
    selected = np.sort(np.concatenate([positive, hard, random]))
    result = data.iloc[selected].copy()
    result["target"] = labels[selected].astype(np.uint8)
    
    return result

In [ ]:
import time
# генерирую обучающий файл для катбуст ранкера (training_pairs.tsv)
train_global, train_local = semantic_candidates(train_queries, allowed_all)
training_path = output_dir / 'training_pairs.tsv'
columns_path = output_dir / 'training_columns.cd'
training_columns = ['target', 'group_id'] + all_features
numeric_features = [name for name in all_features if name not in categorical_features]
n_training_pairs = 0
n_training_groups = 0
preparation_started = time.monotonic()

# Группы записываются подряд: катбуст должен видеть все пары запроса вместе
# Файл открывается с 'w', поэтому повторный запуск не дописывает старые строки
with training_path.open('w', encoding='utf-8', newline='') as stream:
    pd.DataFrame(columns=training_columns).to_csv(stream, sep='\t', index=False)
    for start in tqdm(range(0, len(train_queries), 128), desc='Обучающие пары'):
        parts = []
        for i in range(start, min(start + 128, len(train_queries))):
            query = train_queries.iloc[i]
            data = make_features(query, train_global[i], train_local[i], allowed_all)
            expanded = add_extra_features(data, query)
            sampled = sample_training_pairs(expanded, query.query_key)
            if len(sampled):
                sampled['group_id'] = i
                parts.append(sampled[training_columns])
                n_training_pairs += len(sampled)
                n_training_groups += 1
                
        if parts:
            block = pd.concat(parts, ignore_index=True)
            # Числовые флаги записываем как 0/1, а не строки True/False.
            block[numeric_features] = block[numeric_features].astype(np.float64)
            block.to_csv(stream, sep='\t', header=False, index=False, float_format='%.17g')
            del block
        if start == 384:
            stream.flush()
            processed = min(start + 128, len(train_queries))
            estimate = (time.monotonic() - preparation_started) * len(train_queries) / processed / 3600
            size = stream.tell() * len(train_queries) / processed / 1024**3
            print(f'Оценка подготовки по первым {processed} запросам: {estimate:.1f} ч, TSV: {size:.1f} ГБ.')
            print('Это не включает последующее обучение CatBoost и BGE.')

assert n_training_pairs > 0 and n_training_groups > 0
# Первые две колонки метка и группа, далее числовые и категориальные признаки
column_description = ['0\tLabel', '1\tGroupId']
for i, name in enumerate(all_features, start=2):
    kind = 'Categ' if name in categorical_features else 'Num'
    column_description.append(f'{i}\t{kind}\t{name}')
columns_path.write_text('\n'.join(column_description) + '\n', encoding='utf-8')

print('Пар:', n_training_pairs, 'Контекстов с обучающими парами:', n_training_groups)
print('Пропущено контекстов без обоих классов:', len(train_queries) - n_training_groups)

del train_global, train_local, parts, data, expanded, sampled
gc.collect()

Обучающие пары:   0%|                      | 4/2654 [01:17<14:20:36, 19.49s/it]

Оценка подготовки по первым 512 запросам: 14.2 ч, TSV: 18.2 ГБ.
Это не включает последующее обучение CatBoost и BGE.


Обучающие пары: 100%|██████████████████▉| 2651/2654 [14:36:28<01:00, 20.09s/it]

Файл получился гигантский, оставлял это на ночь, так что сразу не заметил размер. Для такого не хватит оперативки, придется сделать отбор, выберу 50000 контекстов запроса

In [35]:
from pathlib import Path
import random

output_dir = Path("avito_train_outputs")
source_path = output_dir / "training_pairs.tsv"
subset_path = output_dir / "training_pairs_50k.tsv"
temporary_path = subset_path.with_suffix(".tmp")

# Первый проход, тут собираю ID обучающих групп
# Все строки одной группы идут подряд
groups = []
previous_group = None

print("Собираю список запросов", flush=True)

with source_path.open("rb") as source:
    header = next(source)
    assert header.rstrip(b"\r\n").split(b"\t")[:2] == [
        b"target", b"group_id"
    ]

    for line in source:
        group = line.split(b"\t", 2)[1]
        if group != previous_group:
            groups.append(group)
            previous_group = group

assert len(groups) == len(set(groups)), "Строки групп перемешаны"

# Выбираю группы, а не отдельные пары
selected_groups = set(
    random.Random(46).sample(groups, min(50_000, len(groups)))
)

print(
    f"Выбрано {len(selected_groups):,} из {len(groups):,} контекстов",
    flush=True,
)

# Второй проход, тут переношу все пары выбранных запросов
written_rows = 0
written_groups = set()

with source_path.open("rb") as source, temporary_path.open("wb") as target:
    target.write(next(source))

    for line in source:
        group = line.split(b"\t", 2)[1]
        if group in selected_groups:
            target.write(line)
            written_rows += 1
            written_groups.add(group)

assert written_groups == selected_groups
temporary_path.replace(subset_path)

print(f"Сохранено пар: {written_rows:,}")
print(f"Размер: {subset_path.stat().st_size / 1024**3:.2f} ГБ")
print("Файл:", subset_path)

Собираю список запросов
Выбрано 50,000 из 317,486 контекстов
Сохранено пар: 6,465,716
Размер: 2.88 ГБ
Файл: avito_train_outputs/training_pairs_50k.tsv


In [38]:
# Тут загружаю подготовленный датасет в катбуст, обучаю ранжировщик и сохраняю модель
training_path = output_dir / "training_pairs_50k.tsv"
columns_path = output_dir / 'training_columns.cd'
training_pool = Pool(str(training_path), column_description=str(columns_path), delimiter='\t', has_header=True, thread_count=8)

ranker = CatBoostRanker(
    iterations=100, depth=6, learning_rate=0.05, l2_leaf_reg=10,
    loss_function="YetiRank", random_seed=seed, thread_count=8,
    task_type="CPU", allow_writing_files=False, verbose=100,
)
ranker.fit(training_pool)
ranker.save_model(str(output_dir / "ranker_extended_100.cbm"))
selected_trees = 100

del training_pool
gc.collect()

Groupwise loss function. OneHotMaxSize set to 10
0:	total: 2.09s	remaining: 3m 26s
99:	total: 3m 20s	remaining: 0us


1567

## 8. Отбор 200 кандидатов и BGE
Тут смешиваю места гибрида и катбуст с равными весами:
`0.5 / (60 + место_гибрида) + 0.5 / (60 + место_CatBoost)`.
Первые 200 объявлений передаю BGE реранкеру. BGE совместно читает запрос и текст объявления, максимальная длина пары 512 токенов. Беру готовые веса, дообучения BGE нет.
Затем снова смешиваю места: `75% ранжирования предыдущего этапа + 25% BGE`, с той же константой 60. Выбираю первые 50. Чистый BGE при экспериментах был хуже смеси, поэтому я не заменяю им весь поиск

In [40]:
def shortlist(query, global_ids, local_ids, allowed):
    data = make_features(query, global_ids, local_ids, allowed)
    expanded = add_extra_features(data, query)
    scores = ranker.predict(expanded[all_features], ntree_end=selected_trees)
    rows = data.doc_idx.to_numpy()
    blend = 0.5 / (60 + data.reference_rank.to_numpy()) + 0.5 / (60 + ranks_by_score(rows, scores))
    return rows[order_by_score(rows, blend)[:200]]


# Правило выбора типа совпадает с BGE экспериментом
ce_dtype = torch.float16
ce_batch = 8
CE_MAX_LENGTH = 512
ce_tokenizer = AutoTokenizer.from_pretrained(BGE_PATH, local_files_only=True)
ce_model = AutoModelForSequenceClassification.from_pretrained(
    BGE_PATH, local_files_only=True, torch_dtype=ce_dtype, use_safetensors=True
).to(device).eval()
print('BGE:', ce_dtype)

BGE: torch.float16


In [45]:
def ce_score(query, rows): #инференс модели
    global ce_batch
    query_text = '\n'.join(filter(None, [clean_dense_text(query.search_query),
                                         clean_dense_text(query.search_infm_params_text)]))
    docs = ['\n'.join(filter(None, [clean_dense_text(corpus.iloc[int(row)][col])
            for col in ['item_title_raw', 'item_infm_params_text', 'item_description_raw']]))
            for row in rows]
    scores = []
    start = 0
    
    while start < len(rows):
        batch = docs[start:start + ce_batch]
        try:
            tokens = ce_tokenizer([[query_text, doc] for doc in batch], padding=True,
                truncation=True, max_length=CE_MAX_LENGTH, return_tensors='pt').to('cuda')
            with torch.inference_mode():
                values = ce_model(**tokens, return_dict=True).logits.reshape(-1).float().cpu().numpy()
            scores.extend(values.tolist())
            start += len(batch)
            del tokens
            
        except torch.cuda.OutOfMemoryError:
            if 'tokens' in locals():
                del tokens
            gc.collect()
            torch.cuda.empty_cache()
            if ce_batch == 1:
                raise
            ce_batch = max(1, ce_batch // 2)
            print('Уменьшаем batch size:', ce_batch)
    values = np.asarray(scores, dtype=np.float32)

    return values

In [46]:
def ce_select(rows, scores, weight): #Замешиваю через RRF
    if weight == 0:
        return rows[:50].copy()
    base_rank = np.arange(1, len(rows) + 1)
    neural_rank = ranks_by_score(rows, scores)
    mixed = (1 - weight) / (60 + base_rank) + weight / (60 + neural_rank)
    return rows[order_by_score(rows, mixed)[:50]]

## 9. Проверка качества

Recall@50 это доля известных релевантных объявлений, попавших в первые 50. Усредняю по запросам. Дополнительно считаю покрытие первых 200 кандидатов (`pool_recall`) и верхнюю границу Recall при идеальном выборе 50 из этих 200 (`oracle50`)


In [47]:
def query_metrics(predicted, candidates, relevant): #Расчет метрик и ниже простейшая проверка
    relevant = set(relevant)
    found = len(set(candidates) & relevant)
    return {
        "recall50": len(set(predicted) & relevant) / len(relevant),
        "pool_recall": found / len(relevant),
        "oracle50": min(50, found) / len(relevant),
    }


print(query_metrics([1, 2, 3], [1, 2, 3, 8], [1, 2, 3, 4]))

{'recall50': 0.75, 'pool_recall': 0.75, 'oracle50': 0.75}


In [48]:
def predict_queries(frame, allowed, relevant=None): #инференс всего пайплайна на наборе запросов
    global_ids, local_ids = semantic_candidates(frame, allowed)
    predictions, metrics = [], []
    for i, query in enumerate(tqdm(frame.itertuples(index=False), total=len(frame), desc='CatBoost + BGE')):
        rows = shortlist(query, global_ids[i], local_ids[i], allowed)
        scores = ce_score(query, rows)
        selected = ce_select(rows, scores, 0.25)

        predictions.append(' '.join(item_ids[selected]))
        if relevant is not None:
            metrics.append({'query_key': int(query.query_key), 'fold': int(getattr(query, 'fold', -1)),
                            **query_metrics(selected, rows, relevant[query.query_key])})
    return predictions, pd.DataFrame(metrics)

## 10. Формирование answer.csv

На финальном этапе допустимы только объявления из `benchmark_items.parquet`. Сохраняю исходный порядок benchmark запросов. Все модели работают локально, обращений к внешним API тут нет, валидацию из-за дефицита времени не проверяю

In [49]:
answer_strings, _ = predict_queries(benchmark_queries, allowed_benchmark)
answer = pd.DataFrame({'query_id': benchmark_queries.query_id.astype(str), 'answer': answer_strings})

CatBoost + BGE: 100%|████████████████████| 2452/2452 [1:21:11<00:00,  1.99s/it]


In [52]:
answer.to_csv(output_dir / 'answer.csv', index=False, encoding="utf-8")
display(answer.head(3))

,query_id,answer
0,70DfDUpwjxB4lzFd,355392014208b7bf 9515d1e1ecdac72c 255fbeaf526a1cc1 d722bcda1a555091 603623b4bd9e8f6c b...
1,JTrdTaZJvSiLPkXj,422d3ffdd5bbf626 cbeccbecb1fb8d86 4a435cc0a254c1fd bafe6304f743b5fb 413f6384e3c07b64 8...
2,LZCZNoVG4AFUkVRJ,3b370cc603f67947 168a9207e80b0be4 d8fce513e4f000a7 dab52187b4500d9b edecb3695ab88901 3...


В итоге на скрытом тесте метрика Recall@50: 0.809435, это чуть хуже лучшей отправки, на которой получилось Recall@50: 0.811687. /то связано с изменением количества контекстов запроса. Однако я немного менял логику выбора этих контекстов, поэтому текущий ноутбук воспроизводит финальную версию решения, но не в точности конфигурацию, на которой был получен лучший результат

## 11. Сохранение requirments

Ниже сохраняются все версии библиотек и остальные метаданные запуска

In [60]:
from importlib.metadata import distributions
import platform

versions = sorted({f"{d.metadata['Name']}=={d.version}" for d in distributions() if d.metadata['Name']})
(output_dir / 'requirements_run.txt').write_text('\n'.join(versions) + '\n')
run_info = {
    'seed': seed, 'python': platform.python_version(), 'torch': torch.__version__,
    'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
    'e5_dtype': str(dtype), 'bge_dtype': str(ce_dtype), 'bge_batch_final': ce_batch,
    'e5_revision': model_revision, 'bge_revision': bge_revision,
    'inputs': {name: str(data_dir / name) for name in
               ['train.parquet', 'benchmark_items.parquet', 'benchmark_queries.parquet']},
    'train_keys': train_queries.query_key.astype(str).tolist(),
    'features': all_features, 'used_trees': selected_trees,
    'catboost_weight': 0.5, 'bge_weight': 0.25,
    'training_contexts': len(train_queries),
    'training_texts': int(train_queries.query_group.nunique())
}
(output_dir / 'run_info.json').write_text(json.dumps(run_info, ensure_ascii=False, indent=2))

9306376

## Использованные открытые модели, библиотеки и подходы

- [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base): готовый энкодер текстов. Подготовка префиксов и mean pooling основаны на примере авторов,
- [BGE-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3): готовый реранкер. Получение логитов через Transformers основано на примере авторов,
- [BM25s](https://github.com/xhluca/bm25s): реализация BM25,
- [PyStemmer](https://github.com/snowballstem/pystemmer): основы слов,
- [FAISS](https://github.com/facebookresearch/faiss): точный векторный поиск IndexFlatIP,
- [CatBoost](https://catboost.ai/): CatBoostRanker с готовой функцией потерь YetiRan,
- [PyTorch](https://pytorch.org/) и [Transformers](https://github.com/huggingface/transformers): локальное выполнение нейросетей,
- NumPy, pandas, SciPy, PyArrow, tokenizers, safetensors, SentencePiece, threadpoolctl и tqdm: обработка данных и вычисления. huggingface-hub нужен только для ячейки скачивания весов,
- BM25, косинусный поиск, обучение ранжированию и RRF: готовые подходы. Собственная часть решения это подготовка данных, разбиения, объединение кандидатов, признаки, отбор обучающих пар и проверка комбинации методов

Итог: `answer.csv`. Для автономного воспроизведения также нужны исходные данные, установленные библиотеки и папка `models` с открытыми весами. Обученная модель катбуст сохраняется как результат, но следующий запуск её не читает